# 1. Setup & Imports

In [16]:
# Data handling
import pandas as pd
import numpy as np
import os

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import normalize

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
# for removing genes that aren't substantial or don't offer any insight
from sklearn.feature_selection import VarianceThreshold

# Download Data
import subprocess
subprocess.run(['kaggle', 'datasets', 'download', '-d', 'crawford/gene-expression', '-p', 'data/', '--unzip'])

CompletedProcess(args=['kaggle', 'datasets', 'download', '-d', 'crawford/gene-expression', '-p', 'data/', '--unzip'], returncode=0)

# 2. Data Loading

Below we can see the actual.csv is presenting us 72 patients and whether their cancer is ALL or AML

In [17]:
labels = pd.read_csv('data/actual.csv')
print(labels.head(10))
print(labels.shape)

   patient cancer
0        1    ALL
1        2    ALL
2        3    ALL
3        4    ALL
4        5    ALL
5        6    ALL
6        7    ALL
7        8    ALL
8        9    ALL
9       10    ALL
(72, 2)


## Data Overview

The training dataset contains 7,129 rows/ genes and 78 columns. 

The columns break down as:
- **Gene Description** — the full name of each gene
- **Gene Accession Number** — the unique gene identifier
- **Expression value columns** (e.g. `1`, `2`, `3`...) — the numeric expression level for each patient
- **Call columns** (e.g. `call`, `call.1`...) — present/absent flags indicating if a gene was reliably detected, not needed for our analysis

The data is currently **transposed** from what we need — genes are rows and patients are columns. We need to flip this so that **each row is a patient** and **each column is a gene**, which is the standard format for machine learning.

In [18]:
train = pd.read_csv('data/data_set_ALL_AML_train.csv')
test = pd.read_csv('data/data_set_ALL_AML_independent.csv')

print(train.shape)
print(train.head())

(7129, 78)
                      Gene Description Gene Accession Number    1 call    2  \
0  AFFX-BioB-5_at (endogenous control)        AFFX-BioB-5_at -214    A -139   
1  AFFX-BioB-M_at (endogenous control)        AFFX-BioB-M_at -153    A  -73   
2  AFFX-BioB-3_at (endogenous control)        AFFX-BioB-3_at  -58    A   -1   
3  AFFX-BioC-5_at (endogenous control)        AFFX-BioC-5_at   88    A  283   
4  AFFX-BioC-3_at (endogenous control)        AFFX-BioC-3_at -295    A -264   

  call.1    3 call.2    4 call.3  ...   29 call.33   30 call.34   31 call.35  \
0      A  -76      A -135      A  ...   15       A -318       A  -32       A   
1      A  -49      A -114      A  ... -114       A -192       A  -49       A   
2      A -307      A  265      A  ...    2       A  -95       A   49       A   
3      A  309      A   12      A  ...  193       A  312       A  230       P   
4      A -376      A -419      A  ...  -51       A -139       A -367       A   

    32 call.36   33 call.37  
0 -

# 3. PreProcessing

### Training dataset

Drop the gene description and call columns

In [19]:
# Drop gene description columns and 'call' columns
train_expr = train.drop(columns=['Gene Description', 'Gene Accession Number'])
train_expr = train_expr.loc[:, ~train_expr.columns.str.contains('call')]

print(train_expr.shape)

(7129, 38)


Transpose or flip dataset to format acceptable for model training - also convert categorical or string data to numerical for same reason - in this case AML v ALL classification

In [20]:
# Transpose so rows = patients, columns = genes
train_expr = train_expr.T

# Convert to numeric
train_expr = train_expr.apply(pd.to_numeric, errors='coerce')

print(train_expr.shape)

(38, 7129)


In [22]:
# For training dataset
# Attach labels to training data
train_labels = labels[labels['patient'] <= 38]['cancer'].values

print(train_labels.shape)
print(train_labels)

(38,)
['ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL'
 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL'
 'ALL' 'ALL' 'ALL' 'AML' 'AML' 'AML' 'AML' 'AML' 'AML' 'AML' 'AML' 'AML'
 'AML' 'AML']


### Test Dataset

Drop the gene description and call columns

In [23]:
# Drop gene description columns and 'call' columns
test_expr = test.drop(columns=['Gene Description', 'Gene Accession Number'])
test_expr = test_expr.loc[:, ~test_expr.columns.str.contains('call')]

print(test_expr.shape)

(7129, 34)


Transpose or flip dataset to format acceptable for model training - also convert categorical or string data to numerical for same reason - in this case AML v ALL classification

In [24]:
# Transpose so rows = patients, columns = genes
test_expr = test_expr.T

# Convert to numeric
test_expr = test_expr.apply(pd.to_numeric, errors='coerce')

print(test_expr.shape)

(34, 7129)


In [26]:
# For training dataset
# Attach labels to training data
test_labels = labels[labels['patient'] > 38]['cancer'].values

print(test_labels.shape)
print(test_labels)

(34,)
['ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'AML'
 'AML' 'AML' 'AML' 'AML' 'ALL' 'ALL' 'AML' 'AML' 'ALL' 'AML' 'AML' 'AML'
 'AML' 'AML' 'AML' 'AML' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL' 'ALL']


## Normalisation

In [32]:
# source : https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

print(train_expr.head())
print(train_expr.describe())

scaler = StandardScaler()

print(scaler.fit(train_expr))

print(scaler.mean_)

print(scaler.transform(train_expr))

# using standard scaler to normalise data
train_expr_scaled = scaler.fit_transform(train_expr)
test_expr_scaled = scaler.transform(test_expr)

   0     1     2     3     4     5     6     7     8     9     ...  7119  \
1  -214  -153   -58    88  -295  -558   199  -176   252   206  ...   185   
2  -139   -73    -1   283  -264  -400  -330  -168   101    74  ...   169   
3   -76   -49  -307   309  -376  -650    33  -367   206  -215  ...   315   
4  -135  -114   265    12  -419  -585   158  -253    49    31  ...   240   
5  -106  -125   -76   168  -230  -284     4  -122    70   252  ...   156   

   7120  7121  7122  7123  7124  7125  7126  7127  7128  
1   511  -125   389   -37   793   329    36   191   -37  
2   837   -36   442   -17   782   295    11    76   -14  
3  1199    33   168    52  1138   777    41   228   -41  
4   835   218   174  -110   627   170   -50   126   -91  
5   649    57   504   -26   250   314    14    56   -25  

[5 rows x 7129 columns]
             0           1           2           3           4           5     \
count   38.000000   38.000000   38.000000   38.000000   38.000000   38.000000   
mean  -1

### Variance Thresholding

removing gene expressions that are rarely coming up accross patients - essentially removing non substantial or imformative data 

In [34]:

# X = [[0, 2, 0, 3], [0, 1, 4, 3], [0, 1, 1, 3]]
selector = VarianceThreshold()
selector.fit_transform(train_expr_scaled)

array([[-0.86149567, -0.03310102, -0.3517011 , ...,  0.54606799,
        -0.43582025, -0.25587506],
       [-0.16772267,  1.03740009,  0.13913948, ..., -0.26704265,
        -0.59574421,  0.49964792],
       [ 0.41504666,  1.35855042, -2.49589941, ...,  0.70869012,
        -0.38436645, -0.38727036],
       ...,
       [ 0.82206015,  1.35855042,  0.56970139, ..., -1.4704464 ,
        -0.51647755, -0.09163093],
       [-0.02896807,  0.95711251, -0.1708651 , ...,  0.64364126,
        -0.28702143,  0.86098499],
       [-0.13072144, -0.47468273, -0.45503596, ..., -1.01510444,
         0.397175  ,  0.63104322]])

# 4. Exploratory Data Analysis (PCA)

# 5. Feature Selection

# 6. Classification

# 7. Evaluation & Results

# 8. Conclusion